<a href="https://colab.research.google.com/github/eason-lin0213/programming-language/blob/main/41171109H%E6%9E%97%E6%98%93%E8%BE%B0_HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip install -U -q google-generativeai

In [2]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [19]:
from google.colab import userdata

# 從 Colab Secrets 中獲取 API 金鑰
api_key = userdata.get('gemini')

# 使用獲取的金鑰配置 genai
genai.configure(api_key=api_key)

model = genai.GenerativeModel('gemini-2.5-pro')

In [20]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1TVyyEw-UCQtYXZ5B4meyFBIS5oixC81_zcTYBod0nC0/edit?gid=0#gid=0"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "科目", "作業成績"]

_auth_done = False
_gc = None
_ws = None

In [21]:
# --- 主要功能區塊 ---
def get_user_grades():
    """
    透過終端機輸入學生成績，直到使用者輸入 'q' 結束。
    """
    print("--- 準備輸入成績。輸入 'q' 來停止。---")
    grades = []
    while True:
        subject = input("請輸入科目（或輸入 'q' 停止）：")
        if subject.lower() == 'q':
            break

        grade = input(f"請輸入 {subject} 的成績：")
        try:
            grade = int(grade)
        except ValueError:
            print("成績必須是數字。請重新輸入。")
            continue

        today = datetime.now().strftime('%Y-%m-%d')
        grades.append([today, subject, grade])
        print(f"已記錄：日期: {today}, 科目: {subject}, 成績: {grade}\n")

    return grades

In [30]:
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    model = genai.GenerativeModel('gemini-flash-latest')

    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = model.generate_content(prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"

In [23]:
new_grades = get_user_grades()

--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：微積分
請輸入 微積分 的成績：95
已記錄：日期: 2026-03-25, 科目: 微積分, 成績: 95

請輸入科目（或輸入 'q' 停止）：工程數學
請輸入 工程數學 的成績：96
已記錄：日期: 2026-03-25, 科目: 工程數學, 成績: 96

請輸入科目（或輸入 'q' 停止）：資料結構
請輸入 資料結構 的成績：94
已記錄：日期: 2026-03-25, 科目: 資料結構, 成績: 94

請輸入科目（或輸入 'q' 停止）：程式語言
請輸入 程式語言 的成績：95
已記錄：日期: 2026-03-25, 科目: 程式語言, 成績: 95

請輸入科目（或輸入 'q' 停止）：q


In [24]:
new_grades

[['2026-03-25', '微積分', 95],
 ['2026-03-25', '工程數學', 96],
 ['2026-03-25', '資料結構', 94],
 ['2026-03-25', '程式語言', 95]]

In [31]:
get_ai_summary(new_grades)


--- 正在呼叫 AI 模型生成摘要... ---


'根據您提供的成績列表，這是一位在數學理論與資訊科學領域表現皆極為優異且均衡的學生。以下是針對這些成績的摘要與相關學科的常見迷思整理：\n\n### 一、 成績摘要\n\n*   **整體表現：** 該學生在所有科目中展現了高度的一致性，成績落點極為集中（94分至96分之間），平均分數約為 95 分。\n*   **學科均衡性：**\n    *   **數學領域：** 在「微積分」與「工程數學」兩門高度抽象且需要嚴謹邏輯的科目中取得高分（95、96），顯示其具備強大的運算能力與邏輯推導基礎。\n    *   **資訊領域：** 在「資料結構」與「程式語言」兩門核心專業科目中同樣表現優異（94、95），顯示其能將抽象邏輯轉化為具體的演算法實作與系統理解。\n*   **總結：** 該生具備優秀的邏輯整合能力，能同時駕馭純理論計算與應用實作類的學科，屬於全方位發展的理工人才。\n\n---\n\n### 二、 常見迷思整理\n\n針對這四門難度較高的學科，以下是學習者常有的迷思，而從成績來看，該生顯然已成功克服了這些認知障礙：\n\n#### 1. 數學學科（微積分、工程數學）的迷思：\n*   **迷思一：只要背熟公式就能拿高分。**\n    *   **真相：** 公式只是工具，真正的關鍵在於理解公式的「適用條件」與「物理意義」。特別是在工程數學中，如何將現實問題建模（Modeling）成數學方程式，遠比單純的積分運算重要。\n*   **迷思二：微積分與未來的專業課無關。**\n    *   **真相：** 微積分是所有近代工程學的語言。工程數學、訊號處理、甚至人工智慧中的梯度下降法，全部建立在微積分的基礎之上。\n\n#### 2. 資訊學科（資料結構、程式語言）的迷思：\n*   **迷思三：資料結構就是把資料存起來而已。**\n    *   **真相：** 資料結構的核心在於「時間」與「空間」的效率權衡（Trade-off）。選擇正確的資料結構是為了優化演算法的執行效能，而非僅僅是存放。\n*   **迷思四：學習程式語言就是學習語法（Syntax）。**\n    *   **真相：** 語法只是表象，真正的核心是「程式範式」（如物件導向、函數式編程）以及記憶體管理、編譯原理等底層邏輯。理解「為什麼要這樣設計語言」比「怎麼寫」更重要。\n\n##

In [33]:
def main():
    """
    主程式流程：輸入成績 -> 獲取 AI 摘要 -> 寫入 Google Sheet。
    """
    try:
        # 1. Google Sheet 身份驗證
        auth.authenticate_user()

        creds, _ = default()
        gc = gspread.authorize(creds)

        sh = gc.open_by_url(SHEET_URL)
        ws = sh.worksheet(WORKSHEET_NAME)





        print("--- Google Sheet 連線成功。---")

        # 2. 獲取使用者輸入的成績
        new_grades = get_user_grades()

        if not new_grades:
            print("沒有輸入任何成績，程式結束。")
            return

        # 3. 將新成績寫入 Google Sheet
        ws.append_rows(new_grades)
        print("\n--- 成績已成功寫入 Google Sheet。---")

        # 4. 獲取 AI 摘要並寫入 Google Sheet
        summary = get_ai_summary(new_grades)

        # 尋找第一行空白列
        next_row = len(ws.col_values(1)) + 1

        # 使用 update_cell() 方法逐一更新儲存格
        ws.update_cell(next_row, 1, datetime.now().strftime('%Y-%m-%d'))
        ws.update_cell(next_row, 2, 'AI 摘要')

        # 為了避免單元格內容過長，將摘要內容分成多行來寫入
        summary_lines = summary.split('\n')
        for i, line in enumerate(summary_lines):
            ws.update_cell(next_row + i, 3, line)

        print("\n--- AI 摘要已成功寫入 Google Sheet。---")
        print("以下是 AI 生成的摘要內容：")
        print("-" * 50)
        print(summary)
        print("-" * 50)

    except gspread.exceptions.APIError as e:
        print(f"Google Sheets API 錯誤：{e.response.text}")
        print("請確認：")
        print("1. 您的服務帳戶金鑰檔案正確且未過期。")
        print("2. 您已將服務帳戶的 Email 地址（在 JSON 檔案中）分享給 Google Sheet，並給予編輯權限。")
    except Exception as e:
        print(f"發生未預期的錯誤：{e}")

if __name__ == "__main__":
    main()

--- Google Sheet 連線成功。---
--- 準備輸入成績。輸入 'q' 來停止。---
請輸入科目（或輸入 'q' 停止）：微積分
請輸入 微積分 的成績：95
已記錄：日期: 2026-03-25, 科目: 微積分, 成績: 95

請輸入科目（或輸入 'q' 停止）：工程數學
請輸入 工程數學 的成績：96
已記錄：日期: 2026-03-25, 科目: 工程數學, 成績: 96

請輸入科目（或輸入 'q' 停止）：資料結構
請輸入 資料結構 的成績：94
已記錄：日期: 2026-03-25, 科目: 資料結構, 成績: 94

請輸入科目（或輸入 'q' 停止）：程式語言
請輸入 程式語言 的成績：95
已記錄：日期: 2026-03-25, 科目: 程式語言, 成績: 95

請輸入科目（或輸入 'q' 停止）：q

--- 成績已成功寫入 Google Sheet。---

--- 正在呼叫 AI 模型生成摘要... ---

--- AI 摘要已成功寫入 Google Sheet。---
以下是 AI 生成的摘要內容：
--------------------------------------------------
根據您提供的成績列表，以下是針對這些科目的**成績摘要**與**相關學科的常見迷思整理**：

### 一、成績摘要
這份成績單顯示出該生在 **2026年3月25日** 參與的四項考核中，表現極為優異且均衡。
*   **學科領域：** 涵蓋了「數學基礎」（微積分、工程數學）與「資訊核心理論」（資料結構、程式語言）兩大支柱。
*   **表現分析：** 四科成績均落在 **94至96分** 之間，區間極窄，顯示學生在邏輯運算與抽象思維能力上發展非常平均，並未出現偏科現象。
*   **綜合評價：** 展現了強大的數理邏輯能力，並能將其穩定發揮在理論與應用科目上。

---

### 二、常見迷思整理
針對這四門高難度學科，一般學習者常存在的迷思如下：

#### 1. 微積分與工程數學 (數學領域)
*   **迷思一：只要會背公式就能拿高分。**
    *   *事實：* 公式只是工具，真正的關鍵在於理解「變率」與「建模」。若不理解定義（如 $\epsilon-\delta$ 定義或微分方程的物理意義），遇到變形